# Sesión 4 · Matriz de features X e y

Construcción de X (predictores) y y (objetivo) desde `motor_vehicle_limpio_propio.csv` — insumo directo del modelo de IA preventiva de FASE_5.

Ver `sesiones/s4/transformaciones_autogest.py`.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.decomposition import PCA

df = pd.read_csv("../../data_pipeline/data/processed/motor_vehicle_limpio_propio.csv", low_memory=False)
print(df.shape)

<details><summary><b>Salida ejecutada</b></summary>

```text
(2000000, 31)
```
</details>

## 1. Selección de variables

- **Numéricas (6):** Total Cost, Service Duration Hours, Estimated Cost, Mileage at Service, Technician Rating, Tow Distance Miles.
- **Categóricas (2):** Service Type, Urgency Level.
- **Excluidas:** Service ID (id), fechas (temporalidad a manejar aparte), texto libre (descripciones, nombres, ubicaciones) y banderas derivadas (outliers). La variable objetivo no entra en X.

In [ ]:
X_num = df[NUMERICAS].astype(float)
scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_num), columns=NUMERICAS)
print("Medias:", X_std.mean().round(4).to_dict())
print("Desv.:", X_std.std().round(4).to_dict())

<details><summary><b>Salida ejecutada</b></summary>

```text
Medias: todas ≈ 0.0
Desv.: todas = 1.0 (verificación StandardScaler OK)
```
</details>

In [ ]:
X_cat = pd.get_dummies(df[["Service Type","Urgency Level"]], dtype=int)
X = pd.concat([X_std, X_cat], axis=1)
print("X final:", X.shape)

<details><summary><b>Salida ejecutada</b></summary>

```text
X final: (2000000, 16)
```
</details>

## 2. Variable objetivo

`y = 1` si `Follow-up Needed == 'Yes'` (50 % de la población).

In [ ]:
y = df["Follow-up Needed"].eq("Yes").astype(int)
print(y.value_counts(normalize=True).round(3))

<details><summary><b>Salida ejecutada</b></summary>

```text
0    0.5
1    0.5
Name: requiere_seguimiento, dtype: float64
```
</details>

## 3. Discretización y PCA

- Total Cost → 3 rangos de negocio por cuartiles: **económico / medio / costoso**.
- PCA sobre las 6 numéricas estandarizadas: **4 componentes explican el 85%** de la varianza; los primeros 2 solo el 51.7%.

In [ ]:
print(pd.qcut(df["Total Cost"], q=3, labels=["económico","medio","costoso"], duplicates="drop").value_counts())

razones = PCA().fit(X_std).explained_variance_ratio_.cumsum()
print("Varianza acumulada:", razones.round(3).tolist())

<details><summary><b>Salida ejecutada</b></summary>

```text
económico 666,685 | costoso 666,664 | medio 666,651
Varianza acumulada: [0.35, 0.517, 0.684, 0.85, 0.999, 1.0]
```
</details>

## 4. Entregable

Matrices guardadas en `data_pipeline/data/processed/`:
- `X_features.csv` (2,000,000 × 16)
- `y_objetivo.csv` (2,000,000 × 1)

El `StandardScaler` se ajusta solo sobre datos de entrenamiento en producción (evitar fuga).